# Lab: Gradients
## Purpose:
- Automatically compute gradients to incrementally learn decision boundaries.


### Topics:
- Gradient Descent
- JAX
- Decision boundaries

### Steps
* Load the dataset of 2-dimensional embeddings.
* Manually define the model computations and loss function using functions from JAX.
* Automatically compute the gradients of this model.
* Observe how the decision boundary shifts as you are making updates.

Date: 2026-02-24

Source: https://colab.research.google.com/github/google-deepmind/ai-foundations/blob/master/course_3/gdm_lab_3_7_gradients.ipynb

References: https://github.com/google-deepmind/ai-foundations
- GDM GH repo used in AI training courses at the university & college level.

In [ ]:
%%capture
# Install the custom package for this course.
!pip install "git+https://github.com/google-deepmind/ai-foundations.git@main"

# Packages used.
import jax # For automatically computing gradients.
import jax.numpy as jnp # For defining the model and input matrices.
from IPython.display import display # For displaying the dataset.
import pandas as pd # For loading and displaying the dataset.
from ai_foundations import visualizations # For creating plots.

### Load the food/water dataset

In [ ]:
# Load data using pandas.
df = pd.read_csv("https://storage.googleapis.com/dm-educational/assets/ai_foundations/food-water-dataset.csv")

# Extract features (Embedding_dim_1, Embedding_dim_2) and labels.
# Input features (token embeddings).
X_train = jnp.array(df[["Embedding_dim_1", "Embedding_dim_2"]].values)
labels = df["Label"].values # Labels: "food" or "water".

# Convert labels to numeric values for plotting (food = 1, water = 0).
y_train = jnp.where(labels == "food", 1, 0)

# Print and visualize the loaded data for verification.
display(df.head(n=20))

# Shows blue clusters for water/yellow cluster for food
visualizations.plot_data_and_decision_boundary(X_train, labels)

## Initial guess

The weight vector ($\mathbf{w}$) determines the orientation of the decision boundary, and the bias term ($b$) determines how much the decision boundary is shifted.

<br />

------
> **Exercise**
>
>Change the values of $\mathbf{w}$ and $b$ below
and observe how the decision boundary shifts.
>
>Focus on understanding:
>- How the weight vector ($\mathbf{w}$) defines the slope of the decision boundary.
>- How the bias term ($b$) defines the intercept of the decision boundary.
>- What happens when the weight vector points towards the positive class or the negative class.
>
------


In [ ]:
# Change the bias and weights here.
# b_initial_guess = jnp.array([0.5])
# w_initial_guess = jnp.array([-0.9, 0.9])

# b_initial_guess = jnp.array([-0.0])
# w_initial_guess = jnp.array([0.9, 0.9])

b_initial_guess = jnp.array([0.3])
w_initial_guess = jnp.array([0.5, 0.5])

# Plot the initial guess for the weight vector.
visualizations.plot_data_and_decision_boundary(
    X_train,
    labels,
    weight_vector=w_initial_guess,
    bias_term=b_initial_guess,
    title="Initial Guess for Weights"
)

## Implement the prediction function

JAX needs an implementation of the loss function to compute the gradients for a model. The loss function needs an implementation of the neural network that outputs predictions.

------
> **Exercise**
>
> Complete `predict()`. It should implement a single neuron with a sigmoid activation function using tools from `JAX`.
>
> A single neuron makes predictions for an input vector $\mathbf{x}$ as follows:
>
> $$y = \sigma(\mathbf{w} \cdot \mathbf{x} + b)$$
>
> Use the `jnp.dot(a,b)` to compute the dot product between two vectors $\mathbf{a}$ and $\mathbf{b}$.
>
> Use `jax.nn.sigmoid(x)` to compute $\sigma(x)$.
>
> The input for the predict function is a matrix $X$ including the embedding of one prompt in each row, not a vector. Call `jnp.dot` on this matrix to automatically return a vector $\mathbf{y}$ that contains all predictions for all examples represented by $X$. This allows you to jointly compute the predictions for all examples in a dataset rather than computing them one-by-one.
>
> If you do this, however, you will have to change the order of arguments since you are now performing a matrix multiplication between a matrix $X$ of dimension `number_of_examples x embedding_dim` and a matrix $w$ of dimension `embedding_dim x 1`. Unlike for dot products, the order in which you multiply matrices matters and the second dimension of the first matrix and the first dimension of the second matrix have to be the same. Be sure to consider this when you implement the prediction function below.
------

In [ ]:
def predict(w: jax.Array, b: jax.Array, X: jax.Array) -> jax.Array:
    """Computes predicted probabilities using a single neuron.

    This function calculates the dot product of the input features and weights,
    then applies the sigmoid activation function to produce predictions in the
    range [0, 1].

    Args:
        w: The model's weight vector. Expected shape is (embedding_dim,).
        b: The model's bias term. Expected shape is (1,).
        X: The input data matrix where each row is a sample and each column
           is an embedding dimension. Expected shape is
           (number_of_examples, embedding_dim).

    Returns:
        An array of predicted probabilities, one for each sample. The shape
        of the output array is (number_of_examples,).
    """

    y = jax.nn.sigmoid(jnp.dot(X, w) + b)

    return y

## Define the loss function

Use binary cross-entropy loss to train a binary classifier where $y_{\mathrm{pred}}$ is the predicted probability for class 1, and $y_{\mathrm{true}}$ is 1 if the true class is 1 ("food"), and 0 ("water") otherwise:

$$\mathrm{Loss}\left(y_{\mathrm{true}}, \, y_{\mathrm{pred}}\right) = -\left[y_{\mathrm{true}} \log \left(y_{\mathrm{pred}}\right) + \left(1-y_{\mathrm{true}}\right) \log \left(1-y_{\mathrm{pred}}\right)\right]$$

Use `jnp.mean()` to compute the loss across an entire dataset by computing the average loss across all examples.

The loss function below uses the current weights $\mathbf{w}$ and the current bias term $b$ to compute the model predictions on the entire dataset using `predict()`. This makes the loss function directly depend on $\mathbf{w}$ and $b$. The automatic differentiation tools can then automatically compute the gradients for the loss function with respect to $\mathbf{w}$ and $b$.

In [ ]:
def loss_fn(
    w: jax.Array, b: jax.Array, X: jax.Array, y_true: jax.Array
) -> jax.Array:
    """Computes the binary cross-entropy loss.

    This loss function is suitable for binary classification tasks. It
    quantifies the difference between the true binary labels and the
    probabilities predicted by the model. A small epsilon value is added for
    numerical stability to avoid log(0).

    Args:
      w: The model's weight vector. Expected shape is (embedding_dim,).
      b: The model's bias term. Expected shape is (1,).
      X: The input data matrix.
        Expected shape is (number_of_examples, embedding_dim).
      y_true: The true binary labels (0 or 1).
        Expected shape is (number_of_examples,).

    Returns:
      The mean binary cross-entropy loss as a scalar value.
    """
    y_pred = predict(w, b, X)
    # Add a small epsilon for numerical stability.
    epsilon = 1e-8
    return jnp.mean(
        -(
            y_true * jnp.log(y_pred + epsilon)
            + (1 - y_true) * jnp.log(1 - y_pred + epsilon)
        )
    )

Compute the loss for a classifier using the manually set weights on the training dataset.

In [ ]:
# Compute the loss for the initial weights.
print(
    "Loss for current weights:"
    f" {loss_fn(w_initial_guess, b_initial_guess, X_train, y_train):.4f}"
)

## Compute the gradients and update the weights

You will now perform one automatic training step. The cell below uses JAX's `grad` function to define a function that can automatically compute the gradient of the loss function on a dataset.

This gradient tells you how to adjust the weights and the bias terms to reduce the loss.

The `learning_rate` hyperparameter is set to a value smaller than 1.0. The smaller the value, the more the gradient descent can be fine-tuned. *(I suspect Riemann manifolds will be coming into play soon.)*

Compute the updated weight vector by subtracting the gradients from the original weight vectors.

```python
w_updated = w_current - learning_rate * gradient_w
```

and the updated bias term as:

```python
b_updated = b_current - learning_rate * gradient_b
```

Focus on understanding:
- How the gradient points in the direction of the **steepest ascent** (you subtract the gradient to go in the direction of the steepest descent).
- How applying the gradient moves the weights towards better classifying the data.

In [ ]:
# Reset the initial guess.
b_initial_guess = jnp.array([0.5])
w_initial_guess = jnp.array([-0.9, 0.9])

# Define the gradient function that computes the gradient of the loss function.
# `argnums=[0,1]` tells the automatic differentiation method that it should
# compute the gradient with respect to the first and the second argument of the
# loss function, which are the weights and the bias term.
grad = jax.grad(loss_fn, argnums=[0, 1])

# Apply one gradient step.
# Step size for updating the weights.
learning_rate = 0.1

# Compute the gradient.
gradient = grad(w_initial_guess, b_initial_guess, X_train, y_train)
print(f"Gradients: gradient_w = {gradient[0]}, gradient_b = {gradient[1]}")
print(
    f"Loss before update:"
    f" {loss_fn(w_initial_guess, b_initial_guess, X_train, y_train):.4f}"
)

# `gradient` is a tuple that contains the gradient with respect to the weight
# vector and the gradient with respect to the bias term.
gradient_w, gradient_b = gradient

# Update the weights and bias term.
w_updated = w_initial_guess - learning_rate * gradient_w
b_updated = b_initial_guess - learning_rate * gradient_b

print(
    f"Loss after update:"
    f" {loss_fn(w_updated, b_updated, X_train, y_train):.4f}"
)

# Plot the initial guess for the weight vector.
visualizations.plot_data_and_decision_boundary(
    X_train,
    labels,
    weight_vector=w_updated,
    bias_term=b_updated,
    title="Decision Boundary After One Gradient Step",
)

### Gradient descent

The gradient descent algorithm repeatedly performs updates to the weight vector and the bias term using the gradient update formulas from above:

```python
w = w - learning_rate * gradient_w
```

and

```python
b = b - learning_rate * gradient_b
```

The weights `w` and bias term `b` and need to be initialized at the beginning of the training process.

The intialization is usually done randomly in practice, but it's initialized with the values in `w_initial_guess` and `b_initial_guess` here.

Each iteration of this training loop corresponds to one epoch. The training loss decreases with each iteration, and the model becomes better at making predictions.

Focus on observing:
- How the decision boundary shifts closer to separating the two classes.
- How the weight vector gradually aligns with the positive class.

```
The decision will rotate when done correctly.
```

In [ ]:
# Intialize the weight vector.
w = w_initial_guess
b = b_initial_guess
learning_rate = 0.5

for step in range(10):  # Perform 10 update steps.

    # Compute the gradients.
    gradient = grad(w, b, X_train, y_train)
    gradient_w, gradient_b = gradient

    # Update the weights and bias term.
    w = w - learning_rate * gradient_w
    b = b - learning_rate * gradient_b

    print(f"Step {step + 1} loss: {loss_fn(w, b, X_train, y_train):.4f}")

    visualizations.plot_data_and_decision_boundary(
        X_train,
        labels,
        weight_vector=w,
        bias_term=b,
        title=f"Step {step + 1}"
    )

## More complex machine learning problems

------
> **💭 Reflection:**
>
> Run the following cell to plot the spiral dataset that contains datapoints separated into four classes. Then reflect upon the models and methods you have learned in this and previous activities.
>
>* How would you extend the logistic regression model to support more than two classes (e.g., four in this case)?
>
>   * I don't know.
>
>* Could a classifier with a single activation function such as a logistic regression model properly separate this data? Why or why not?
>
>   * Absolutely not. A single regression function would only be able to plot a straight line. The multiple curves in this function are created when the output of one function is adjusted by the next function.
-----



In [ ]:
visualizations.plot_spiral_data(
    "https://storage.googleapis.com/dm-educational/assets/ai_foundations/spiral-dataset.csv"
)